# PerioVision AI: train the PANORAMIC models on a free Colab GPU

**How to use:** `Runtime` → `Change runtime type` → **T4 GPU** (or better) → Save, then `Runtime` → **Run all**
and allow Google Drive. Everything is downloaded from the public sources by the notebook itself.
If Colab disconnects, just **Run all** again: finished steps are skipped and training resumes from Drive.

| Model | Data (downloaded automatically) | Tested on |
|---|---|---|
| A. Bone-loss screen per jaw (yes / no) | [ToothXpert MM-OPG](https://huggingface.co/datasets/jeffrey423/ToothXpert.MM-OPG-Annotations), ~9,200 films | its official 450-film test split |
| B. Worst-tooth bone loss % + stage | [BRAR](https://doi.org/10.6084/m9.figshare.30155974), 988 films (fine-tuned from A) | held-out 15 % of BRAR |
| C. Per-tooth bone-loss detector | [PDCNN](https://github.com/PuckBlink/PDCNN), 1,747 films | held-out 10 % of PDCNN |
| D. Tooth detector fine-tune (FDI) | [Aga Khan OPG](https://zenodo.org/records/10538750), 152 films | held-out 15 %; kept only if better than the current detector |

**Before running D:** upload the current detector `backend/weights/dental_yolov8n.pt` to `MyDrive/PerioVision/current/`.
Without it, step D is skipped.

At the end, download `MyDrive/PerioVision/export_panoramic/` and give it to Claude: every model comes with
its held-out test metrics, and the app only switches a model on if those metrics pass.

In [ ]:
# 0. Settings (defaults chosen for a free T4 GPU)
# Which models to train in this session. Finished models are skipped anyway, so a short session can do one at a time.
RUN = {"A": True, "B": True, "C": True, "D": False}   # D (tooth-detector fine-tune) can also be trained on a CPU
ARCH        = "convnext_tiny"   # backbone for A and B: resnet18 | resnet50 | efficientnet_b3 | convnext_tiny
SIZE        = (1024, 512)       # panoramic film resized to W x H for A and B
EPOCHS_A, EPOCHS_B = 12, 30
BATCH_A, BATCH_B   = 16, 16
# C: per-tooth bone-loss detector. "yolo11s" at 1024 fits one free session (~1-1.5 h on a T4);
# with more GPU time use "yolo11m.pt", 1280, 150 for the most accuracy.
YOLO_C, IMGSZ_C, EPOCHS_C, PATIENCE_C = "yolo11s.pt", 1024, 60, 15
IMGSZ_D, EPOCHS_D = 1280, 40

In [ ]:
# 1. GPU + Google Drive
import subprocess, os, glob, json, shutil
if not shutil.which("nvidia-smi"):
    raise RuntimeError("NO GPU attached. Runtime > Change runtime type > T4 GPU > Save, then Run all again. "
                       "If Colab says 'Cannot connect to GPU backend', this account's free GPU time is used up "
                       "for today: try again later or use another Google account.")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/PerioVision"
EXPORT = f"{ROOT}/export_panoramic"
for d in ("runs", "current", "export_panoramic"):
    os.makedirs(f"{ROOT}/{d}", exist_ok=True)

In [ ]:
# 2. Same Ultralytics version as the PerioVision backend, plus the Google-Drive downloader
!pip -q install "ultralytics==8.4.21" gdown
import ultralytics; ultralytics.checks()

In [ ]:
%%writefile /content/train_panoramic_boneloss.py
"""Panoramic whole-film bone-loss models (CPU-trainable), validated on held-out data.

Stage 1  --task screen : generalised crestal bone loss yes/no for the maxilla and the mandible, trained on
         ToothXpert MM-OPG (huggingface.co/datasets/jeffrey423/ToothXpert.MM-OPG-Annotations, ~9,200 films,
         Apache-2.0), tested on its official held-out test split (450 films).
Stage 2  --task severity : worst-tooth radiographic bone loss (% of root length) regressed from the whole film,
         fine-tuned from the stage-1 network on BRAR (figshare 10.6084/m9.figshare.30155974, 988 films,
         CC BY 4.0); split 70 / 15 / 15 by patient, stratified by BRAR level; the 15 % test split is never used
         for training or model selection. The validation split also sets a split-conformal interval.

Network: ImageNet-pretrained ResNet-18 (torchvision), grey film resized to 512 x 256, replicated to 3 channels.
Outputs (to --out): weights (.pt state_dict) and a metrics JSON. Nothing here is used by the app until its
test metrics pass the bar in docs/MODEL_CARD.md.

Usage (from backend/):
  python scripts/train_panoramic_boneloss.py --task screen --data ~/Downloads/MM-OPG --out weights/panoramic_screen
  python scripts/train_panoramic_boneloss.py --task severity --data ~/Downloads/BRAR/data \\
         --init weights/panoramic_screen.pt --out weights/panoramic_severity
"""
import argparse
import os as _os

_os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # before torch initialises CUDA
import json
import math
import os
import random
import time
import zipfile

import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision

W, H = 512, 256                   # overridden by --size
MEAN, STD = 0.449, 0.226          # ImageNet grey mean / std
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def prep(gray: np.ndarray) -> np.ndarray:
    gray = gray.reshape(gray.shape[:2])
    return cv2.resize(gray, (W, H), interpolation=cv2.INTER_AREA)


def to_tensor(batch: np.ndarray) -> torch.Tensor:
    x = torch.from_numpy(np.ascontiguousarray(batch)).to(DEVICE).float().div_(255.0).sub_(MEAN).div_(STD)
    return x.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()


def augment(img: np.ndarray, rng: random.Random) -> np.ndarray:
    if rng.random() < 0.5:
        img = img[:, ::-1]                         # left-right mirror keeps both jaw labels
    a, b = rng.uniform(0.8, 1.2), rng.uniform(-20, 20)
    img = np.clip(img.astype(np.float32) * a + b, 0, 255)
    m = cv2.getRotationMatrix2D((W / 2, H / 2), rng.uniform(-4, 4), rng.uniform(0.95, 1.05))
    m[:, 2] += (rng.uniform(-12, 12), rng.uniform(-8, 8))
    return cv2.warpAffine(img, m, (W, H), borderMode=cv2.BORDER_REFLECT).astype(np.uint8)


ARCH = "resnet18"                 # overridden by --arch


def network(outputs: int) -> nn.Module:
    """ImageNet-pretrained backbone with a fresh head of `outputs` units (attribute `head` for re-heading)."""
    m = torchvision.models
    if ARCH == "resnet18":
        net = m.resnet18(weights=m.ResNet18_Weights.IMAGENET1K_V1)
        net.fc = nn.Linear(net.fc.in_features, outputs)
    elif ARCH == "resnet50":
        net = m.resnet50(weights=m.ResNet50_Weights.IMAGENET1K_V2)
        net.fc = nn.Linear(net.fc.in_features, outputs)
    elif ARCH == "efficientnet_b3":
        net = m.efficientnet_b3(weights=m.EfficientNet_B3_Weights.IMAGENET1K_V1)
        net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, outputs)
    elif ARCH == "convnext_tiny":
        net = m.convnext_tiny(weights=m.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, outputs)
    else:
        raise ValueError(ARCH)
    return net.to(DEVICE)


def rehead(net: nn.Module, outputs: int) -> nn.Module:
    if hasattr(net, "fc"):
        net.fc = nn.Linear(net.fc.in_features, outputs)
    else:
        net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, outputs)
    return net.to(DEVICE)


def auc(y, p) -> float:
    y, p = np.asarray(y), np.asarray(p)
    pos, neg = p[y == 1], p[y == 0]
    if not len(pos) or not len(neg):
        return float("nan")
    order = np.argsort(np.concatenate([pos, neg]))
    ranks = np.empty(len(order)); ranks[order] = np.arange(1, len(order) + 1)
    return float((ranks[: len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))


# ---------------------------------------------------------------- data
def load_mmopg(folder: str):
    labels = {}
    for split, name in (("train", "mmopg_train.json"), ("test", "mmopg_test.json")):
        for it in json.load(open(os.path.join(folder, name), encoding="utf-8")):
            q = it["conversations"][0]["value"].lower()
            if "generalised crestal bone loss" not in q:
                continue
            jaw = 0 if "maxill" in q else 1
            yes = it["conversations"][1]["value"].strip().lower().startswith("yes")
            labels.setdefault((split, it["image"]), [None, None])[jaw] = int(yes)
    zf = zipfile.ZipFile(os.path.join(folder, "images.zip"))
    names = {os.path.basename(n): n for n in zf.namelist() if n.lower().endswith((".jpg", ".png", ".jpeg"))}
    out = {"train": [], "test": []}
    for (split, img), (mx, md) in labels.items():
        if mx is None or md is None or img not in names:
            continue
        g = cv2.imdecode(np.frombuffer(zf.read(names[img]), np.uint8), cv2.IMREAD_GRAYSCALE)
        if g is not None:
            out[split].append((prep(g), np.array([mx, md], np.float32), img))
    return out


def load_brar(folder: str):
    import pandas as pd

    meta = pd.read_csv(os.path.join(folder, "meta_data.csv"))
    rows = []
    for _, m in meta.iterrows():
        path = os.path.join(folder, f"level_{int(m['Level'])}", m["File name"])
        g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if g is not None:
            rows.append((prep(g), np.array([float(m["Bone resorption"]) * 100.0], np.float32), m["File name"],
                         int(m["Level"]), int(m["Age"])))
    rng = random.Random(0)  # audit-ok: seeded patient-level split
    split = {"train": [], "val": [], "test": []}
    for lv in (1, 2, 3):
        group = [r for r in rows if r[3] == lv]
        rng.shuffle(group)
        n = len(group)
        split["train"] += group[: int(0.7 * n)]
        split["val"] += group[int(0.7 * n): int(0.85 * n)]
        split["test"] += group[int(0.85 * n):]
    return split


# ---------------------------------------------------------------- training
def run_epochs(net, train, val_fn, loss_fn, epochs, lr, batch=32, seed=0, ckpt=None):
    """Train; keep the best epoch by validation score. With `ckpt`, state is saved after every epoch and a
    rerun resumes from it (Colab disconnects lose nothing but the current epoch)."""
    rng = random.Random(seed)  # audit-ok: seeded shuffling / augmentation
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    steps = epochs * math.ceil(len(train) / batch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.15)
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    best, best_state, start = None, None, 0
    if ckpt and os.path.exists(ckpt):
        c = torch.load(ckpt, map_location=DEVICE, weights_only=False)
        net.load_state_dict(c["net"]); opt.load_state_dict(c["opt"]); sched.load_state_dict(c["sched"])
        scaler.load_state_dict(c["scaler"]); rng.setstate(c["rng"])
        best, best_state, start = c["best"], c["best_state"], c["epoch"]
        print(f"resuming after epoch {start} (best val {best:.4f})", flush=True)
    for ep in range(start, epochs):
        net.train()
        rng.shuffle(train)
        t0, tot = time.time(), 0.0
        for i in range(0, len(train), batch):
            chunk = train[i:i + batch]
            x = to_tensor(np.stack([augment(r[0], rng) for r in chunk]))
            y = torch.from_numpy(np.stack([r[1] for r in chunk])).to(DEVICE)
            with torch.autocast(DEVICE, dtype=torch.float16 if DEVICE == "cuda" else torch.bfloat16, enabled=DEVICE == "cuda"):
                out = net(x)
            loss = loss_fn(out.float(), y)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item() * len(chunk)
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        score = val_fn(net)
        print(f"epoch {ep + 1}/{epochs} loss {tot / len(train):.4f} val {score:.4f} ({time.time() - t0:.0f}s)", flush=True)
        if best is None or score > best:
            best, best_state = score, {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        if ckpt:
            torch.save({"net": net.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "rng": rng.getstate(), "best": best,
                        "best_state": best_state, "epoch": ep + 1}, ckpt + ".tmp")
            os.replace(ckpt + ".tmp", ckpt)
    net.load_state_dict(best_state)
    return best


@torch.inference_mode()
def predict(net, rows, batch=8):
    """Small batches in half precision: full-size panoramic films at 1024 x 512 need a lot of GPU memory."""
    net.eval()
    out = []
    for i in range(0, len(rows), batch):
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
            y = net(to_tensor(np.stack([r[0] for r in rows[i:i + batch]])))
        out.append(y.float().cpu().numpy())
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(out)


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--task", choices=("screen", "severity"), required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--init")
    ap.add_argument("--epochs", type=int, default=8)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--arch", default="resnet18", choices=("resnet18", "resnet50", "efficientnet_b3", "convnext_tiny"))
    ap.add_argument("--size", type=int, nargs=2, default=(512, 256), metavar=("W", "H"))
    ap.add_argument("--batch", type=int, default=32)
    args = ap.parse_args()
    global W, H, ARCH
    W, H, ARCH = args.size[0], args.size[1], args.arch
    torch.manual_seed(0)
    data = os.path.expanduser(args.data)

    if args.task == "screen":
        d = load_mmopg(data)
        rng = random.Random(1)  # audit-ok: seeded validation hold-out
        rng.shuffle(d["train"])
        nval = len(d["train"]) // 10
        val, train, test = d["train"][:nval], d["train"][nval:], d["test"]
        print(f"MM-OPG films: train {len(train)}, val {len(val)}, test {len(test)}")
        net = network(2)
        crit = nn.BCEWithLogitsLoss()

        def val_auc(n):
            p = predict(n, val)
            y = np.stack([r[1] for r in val])
            return float(np.nanmean([auc(y[:, j], p[:, j]) for j in (0, 1)]))

        run_epochs(net, train, val_auc, crit, args.epochs, args.lr, args.batch, ckpt=os.path.expanduser(args.out) + ".ckpt")
        pv, yv = 1 / (1 + np.exp(-predict(net, val))), np.stack([r[1] for r in val])
        pt, yt = 1 / (1 + np.exp(-predict(net, test))), np.stack([r[1] for r in test])
        metrics = {"task": "generalised crestal bone loss, per jaw", "dataset": "ToothXpert MM-OPG (Apache-2.0)",
                   "train_films": len(train), "val_films": len(val), "test_films": len(test), "jaws": {}}
        for j, jaw in enumerate(("maxilla", "mandible")):
            thr = max(np.linspace(0.05, 0.95, 91), key=lambda t: ((pv[:, j] >= t) & (yv[:, j] == 1)).sum() / max(1, (yv[:, j] == 1).sum())
                      + ((pv[:, j] < t) & (yv[:, j] == 0)).sum() / max(1, (yv[:, j] == 0).sum()))   # Youden on val
            pred = pt[:, j] >= thr
            pos, neg = yt[:, j] == 1, yt[:, j] == 0
            metrics["jaws"][jaw] = {"test_auc": round(auc(yt[:, j], pt[:, j]), 4), "threshold_from_val": round(float(thr), 3),
                                    "test_sensitivity": round(float(pred[pos].mean()), 4),
                                    "test_specificity": round(float((~pred[neg]).mean()), 4),
                                    "test_accuracy": round(float((pred == pos).mean()), 4),
                                    "test_prevalence": round(float(pos.mean()), 4)}
    else:
        d = load_brar(data)
        train, val, test = d["train"], d["val"], d["test"]
        print(f"BRAR films: train {len(train)}, val {len(val)}, test {len(test)}")
        net = network(2)
        if args.init:
            net.load_state_dict(torch.load(os.path.expanduser(args.init), map_location=DEVICE))
        net = rehead(net, 1)
        crit = nn.SmoothL1Loss(beta=5.0)

        def val_neg_mae(n):
            return -float(np.mean(np.abs(predict(n, val)[:, 0] - np.array([r[1][0] for r in val]))))

        run_epochs(net, train, val_neg_mae, crit, args.epochs, args.lr, args.batch, ckpt=os.path.expanduser(args.out) + ".ckpt")
        st = lambda p: "I" if p < 15 else "II" if p <= 33 else "III"  # noqa: E731
        pv = np.clip(predict(net, val)[:, 0], 0, 100); yv = np.array([r[1][0] for r in val])
        pt = np.clip(predict(net, test)[:, 0], 0, 100); yt = np.array([r[1][0] for r in test])
        scores = np.sort(np.abs(pv - yv))
        q = float(scores[min(len(scores) - 1, math.ceil((len(scores) + 1) * 0.9) - 1)])   # split-conformal 90 %
        sets = [[s for s, (a, b) in {"I": (0, 15), "II": (15, 33), "III": (33, 100)}.items()
                 if min(100, p + q) >= a and max(0, p - q) <= b] for p in pt]
        mean_only = float(np.mean([r[1][0] for r in train]))
        metrics = {"task": "worst-tooth radiographic bone loss % from the whole panoramic film",
                   "dataset": "BRAR (CC BY 4.0), expert worst-tooth bone loss / root length",
                   "train_films": len(train), "val_films": len(val), "test_films": len(test),
                   "test_MAE": round(float(np.mean(np.abs(pt - yt))), 3),
                   "test_median_abs_error": round(float(np.median(np.abs(pt - yt))), 3),
                   "test_MAE_predicting_training_mean": round(float(np.mean(np.abs(mean_only - yt))), 3),
                   "test_stage_agreement": round(float(np.mean([st(a) == st(b) for a, b in zip(pt, yt)])), 4),
                   "test_grade_agreement": round(float(np.mean([
                       (("A" if a / g < 0.25 else "B" if a / g <= 1 else "C") == ("A" if b / g < 0.25 else "B" if b / g <= 1 else "C"))
                       for a, b, g in zip(pt, yt, [r[4] for r in test])])), 4),
                   "conformal_q90_from_val": round(q, 3),
                   "test_interval_coverage": round(float(np.mean(np.abs(pt - yt) <= q)), 4),
                   "test_reference_stage_in_set": round(float(np.mean([st(b) in s for b, s in zip(yt, sets)])), 4),
                   "test_set_sizes": {str(k): sum(len(s) == k for s in sets) for k in (1, 2, 3)}}
    out = os.path.expanduser(args.out)
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    metrics.update({"arch": ARCH, "input_size": [W, H]})
    torch.save(net.state_dict(), out + ".pt")
    with open(out + "_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    if os.path.exists(out + ".ckpt"):
        os.remove(out + ".ckpt")              # training finished: the resume checkpoint is no longer needed
    print(json.dumps(metrics, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile /content/convert_aku_labelme.py
"""Aga Khan University OPG dataset (Zenodo 10.5281/zenodo.10538750, CC BY 4.0) -> YOLO detect, FDI classes.

LabelMe polygons (crown + root) with the FDI number in `group_id` become boxes whose class index matches
the PerioVision tooth detector exactly: 11-18 -> 0-7, 21-28 -> 8-15, 31-38 -> 16-23, 41-48 -> 24-31.
Films are split 70 / 15 / 15 (seeded) so no film appears in two splits; the test split is kept for the
before / after comparison and never used for training.

Usage:  python convert_aku_labelme.py --src <.../dataset> --out <folder>
"""
import argparse
import glob
import json
import os
import random
import shutil

FDI = [f"{q}{n}" for q in (1, 2, 3, 4) for n in range(1, 9)]


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True)
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    anns = sorted(glob.glob(os.path.join(os.path.expanduser(args.src), "*", "annotations", "*.json")))
    random.Random(0).shuffle(anns)  # audit-ok: seeded film-level split
    n = len(anns)
    counts = {"train": 0, "val": 0, "test": 0}
    for i, ann in enumerate(anns):
        split = "train" if i < 0.7 * n else "val" if i < 0.85 * n else "test"
        d = json.load(open(ann, encoding="utf-8"))
        stem = os.path.splitext(os.path.basename(ann))[0]
        imgs = glob.glob(os.path.join(os.path.dirname(ann), "..", "OPGs", stem + ".*"))
        if not imgs:
            continue
        w, h = d["imageWidth"], d["imageHeight"]
        lines = []
        for s in d["shapes"]:
            fdi = str(s.get("group_id"))
            if fdi not in FDI or s.get("shape_type") != "polygon":
                continue
            xs, ys = [p[0] for p in s["points"]], [p[1] for p in s["points"]]
            x1, x2, y1, y2 = max(0, min(xs)), min(w, max(xs)), max(0, min(ys)), min(h, max(ys))
            lines.append(f"{FDI.index(fdi)} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}")
        if not lines:
            continue
        name = f"{os.path.basename(os.path.dirname(os.path.dirname(ann))).replace(' ', '_')}_{stem}"
        for sub in ("images", "labels"):
            os.makedirs(os.path.join(args.out, sub, split), exist_ok=True)
        shutil.copy(imgs[0], os.path.join(args.out, "images", split, name + os.path.splitext(imgs[0])[1]))
        with open(os.path.join(args.out, "labels", split, name + ".txt"), "w") as f:
            f.write("\n".join(lines) + "\n")
        counts[split] += 1
    with open(os.path.join(args.out, "aku.yaml"), "w") as f:
        f.write(f"path: {os.path.abspath(args.out)}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n"
                + "".join(f"  {i}: '{c}'\n" for i, c in enumerate(FDI)))
    print(counts)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile /content/convert_pdcnn_coco.py
"""PDCNN panoramic periodontitis dataset (github.com/PuckBlink/PDCNN, 1,747 films) -> YOLO detect.

The dataset ships COCO exports from the VIA annotator: `via_export_coco_BL.json` (per-tooth boxes labelled for
periodontal bone loss) and `via_export_coco_FI.json` (furcation involvement). This converts ONE of them
(--json) into YOLO boxes, keeping the categories found in the file (printed with their counts so the
meaning of each class can be checked), and splits films 80 / 10 / 10 (seeded) into train / val / test.

Usage:  python convert_pdcnn_coco.py --json via_export_coco_BL.json --images <folder with the films> --out <folder>
"""
import argparse
import collections
import glob
import json
import os
import random
import shutil

import cv2


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--json", required=True)
    ap.add_argument("--images", required=True)
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    coco = json.load(open(args.json, encoding="utf-8"))
    cats = sorted(coco.get("categories", []), key=lambda c: c["id"])
    if not cats:   # VIA sometimes omits categories: fall back to the ids used
        cats = [{"id": i, "name": f"class_{i}"} for i in sorted({a["category_id"] for a in coco["annotations"]})]
    index = {c["id"]: i for i, c in enumerate(cats)}
    print("categories:", [(c["id"], c["name"]) for c in cats])
    print("boxes per category:", collections.Counter(a["category_id"] for a in coco["annotations"]))
    files = {os.path.basename(p).lower(): p for p in glob.glob(os.path.join(args.images, "**", "*"), recursive=True)
             if p.lower().endswith((".jpg", ".jpeg", ".png"))}
    by_image = collections.defaultdict(list)
    for a in coco["annotations"]:
        by_image[a["image_id"]].append(a)
    images = [im for im in coco["images"] if os.path.basename(im["file_name"]).lower() in files and by_image[im["id"]]]
    random.Random(0).shuffle(images)  # audit-ok: seeded film-level split
    n, counts = len(images), collections.Counter()
    for i, im in enumerate(images):
        split = "train" if i < 0.8 * n else "val" if i < 0.9 * n else "test"
        src = files[os.path.basename(im["file_name"]).lower()]
        w, h = im.get("width"), im.get("height")
        if not w or not h:
            h, w = cv2.imread(src, cv2.IMREAD_GRAYSCALE).shape[:2]
        lines = []
        for a in by_image[im["id"]]:
            x, y, bw, bh = a["bbox"]
            if bw <= 1 or bh <= 1:
                continue
            lines.append(f"{index[a['category_id']]} {(x + bw / 2) / w:.6f} {(y + bh / 2) / h:.6f} {bw / w:.6f} {bh / h:.6f}")
        if not lines:
            continue
        for sub in ("images", "labels"):
            os.makedirs(os.path.join(args.out, sub, split), exist_ok=True)
        stem = f"pdcnn_{im['id']}"
        shutil.copy(src, os.path.join(args.out, "images", split, stem + os.path.splitext(src)[1]))
        with open(os.path.join(args.out, "labels", split, stem + ".txt"), "w") as f:
            f.write("\n".join(lines) + "\n")
        counts[split] += 1
    with open(os.path.join(args.out, "pdcnn.yaml"), "w") as f:
        f.write(f"path: {os.path.abspath(args.out)}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n"
                + "".join(f"  {i}: '{c['name']}'\n" for i, c in enumerate(cats)))
    print("films per split:", dict(counts))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
# 3. Download the four datasets (skipped if already present)
import urllib.request, zipfile
D = "/content/data"; os.makedirs(D, exist_ok=True)
def fetch(url, path):
    if not os.path.exists(path):
        print("downloading", url); urllib.request.urlretrieve(url, path)
    return path

need_A = RUN["A"] and not os.path.exists(f"{EXPORT}/panoramic_screen_metrics.json")
need_B = RUN["B"] and not os.path.exists(f"{EXPORT}/panoramic_severity_metrics.json")
need_C = RUN["C"] and not os.path.exists(f"{EXPORT}/pdcnn_boneloss_metrics.json")
need_D = RUN["D"] and not os.path.exists(f"{EXPORT}/aku_detector_finetune_metrics.json")
print("still to train:", [k for k, v in (("A", need_A), ("B", need_B), ("C", need_C), ("D", need_D)) if v])

# MM-OPG (Hugging Face, Apache-2.0) - only needed for A
if need_A:
    os.makedirs(f"{D}/mmopg", exist_ok=True)
    HF = "https://huggingface.co/datasets/jeffrey423/ToothXpert.MM-OPG-Annotations/resolve/main"
    fetch(f"{HF}/MM-OPG-Train-Raw.json", f"{D}/mmopg/mmopg_train.json")
    fetch(f"{HF}/MM-OPG-Test.json", f"{D}/mmopg/mmopg_test.json")
    fetch(f"{HF}/images_resized_public.zip", f"{D}/mmopg/images.zip")

# BRAR (figshare, CC BY 4.0) - only needed for B
if need_B and not os.path.exists(f"{D}/brar/meta_data.csv"):
    zipfile.ZipFile(fetch("https://ndownloader.figshare.com/files/58062268", f"{D}/brar.zip")).extractall(f"{D}/brar")
    inner = glob.glob(f"{D}/brar/**/meta_data.csv", recursive=True)[0]
    if os.path.dirname(inner) != f"{D}/brar":
        for p in os.listdir(os.path.dirname(inner)): shutil.move(os.path.join(os.path.dirname(inner), p), f"{D}/brar")

# Aga Khan OPG (Zenodo, CC BY 4.0) - only needed for D
if need_D and not glob.glob(f"{D}/aku/**/annotations", recursive=True):
    zipfile.ZipFile(fetch("https://zenodo.org/api/records/10538750/files/Niihhaa/Dataset-v1.zip/content", f"{D}/aku.zip")).extractall(f"{D}/aku")

# PDCNN (public Google Drive folder) - only needed for C
if need_C:
    if not glob.glob(f"{D}/pdcnn/**/*.json", recursive=True):
        !gdown --folder "https://drive.google.com/drive/folders/18qUxeRPHPcCQT9o8AgV5400f05Fu3ISW" -O {D}/pdcnn
    for z in glob.glob(f"{D}/pdcnn/**/*.zip", recursive=True):
        if not os.path.isdir(z[:-4]): zipfile.ZipFile(z).extractall(z[:-4])
    print("PDCNN annotation files:", glob.glob(f"{D}/pdcnn/**/*.json", recursive=True))
print(subprocess.run("du -sh " + D + "/*", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 4. Model A: panoramic bone-loss screen per jaw (MM-OPG)
if need_A:
    !python /content/train_panoramic_boneloss.py --task screen --data {D}/mmopg --out {EXPORT}/panoramic_screen --arch {ARCH} --size {SIZE[0]} {SIZE[1]} --epochs {EPOCHS_A} --batch {BATCH_A} --lr 2e-4
if os.path.exists(f"{EXPORT}/panoramic_screen_metrics.json"): print(open(f"{EXPORT}/panoramic_screen_metrics.json").read())

In [ ]:
# 5. Model B: worst-tooth bone loss % and stage (BRAR), fine-tuned from model A
if need_B:
    !python /content/train_panoramic_boneloss.py --task severity --data {D}/brar --init {EXPORT}/panoramic_screen.pt --out {EXPORT}/panoramic_severity --arch {ARCH} --size {SIZE[0]} {SIZE[1]} --epochs {EPOCHS_B} --batch {BATCH_B} --lr 1e-4
if os.path.exists(f"{EXPORT}/panoramic_severity_metrics.json"): print(open(f"{EXPORT}/panoramic_severity_metrics.json").read())

In [ ]:
# 6. Model C: per-tooth bone-loss detector (PDCNN). Resumes from Drive after a disconnect.
from ultralytics import YOLO
if not need_C:
    print("Model C already trained or switched off in RUN - skipping")
else:
    bl = [p for p in glob.glob(f"{D}/pdcnn/**/*.json", recursive=True) if "BL" in os.path.basename(p)]
    assert bl, "PDCNN bone-loss JSON not found - check the printout of step 3"
    if not os.path.exists("/content/pdcnn_yolo/pdcnn.yaml"):
        !python /content/convert_pdcnn_coco.py --json "{bl[0]}" --images {D}/pdcnn --out /content/pdcnn_yolo
    print(open("/content/pdcnn_yolo/pdcnn.yaml").read())
    run_c = f"{ROOT}/runs/pdcnn_bl"
    if os.path.exists(f"{run_c}/weights/last.pt") and not os.path.exists(f"{run_c}/DONE"):
        YOLO(f"{run_c}/weights/last.pt").train(resume=True)
    elif not os.path.exists(f"{run_c}/DONE"):
        YOLO(YOLO_C).train(data="/content/pdcnn_yolo/pdcnn.yaml", imgsz=IMGSZ_C, epochs=EPOCHS_C, patience=PATIENCE_C, batch=-1,
                           project=f"{ROOT}/runs", name="pdcnn_bl", exist_ok=True, seed=0, deterministic=False,
                           fliplr=0.5, mosaic=0.5, close_mosaic=10, cache="ram", save_period=1)
    open(f"{run_c}/DONE", "w").close()

    # Honest test: detection mAP per class + per-tooth label accuracy on the held-out 10 %
    model_c = YOLO(f"{run_c}/weights/best.pt")
    r = model_c.val(data="/content/pdcnn_yolo/pdcnn.yaml", split="test", imgsz=IMGSZ_C, project=f"{ROOT}/runs", name="pdcnn_test", exist_ok=True)
    def iou(a, b):
        x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1); return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)
    right = total = 0
    for img in glob.glob("/content/pdcnn_yolo/images/test/*"):
        lab = img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        p = model_c(img, imgsz=IMGSZ_C, conf=0.25, verbose=False)[0]
        H, W = p.orig_shape
        preds = [(b.xyxy[0].tolist(), int(b.cls)) for b in p.boxes]
        for line in open(lab):
            c, cx, cy, w, h = map(float, line.split()); g = [(cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H]
            best = max(preds, key=lambda q: iou(g, q[0]), default=None)
            total += 1
            if best is not None and iou(g, best[0]) >= 0.5 and best[1] == int(c): right += 1
    names = model_c.names
    mc = {"task": "per-tooth periodontal bone-loss boxes (PDCNN)", "classes": names,
          "test_mAP50": round(float(r.box.map50), 4), "test_mAP50_95": round(float(r.box.map), 4),
          "test_per_class_mAP50": {names[i]: round(float(v), 4) for i, v in enumerate(r.box.all_ap[:, 0])},
          "test_tooth_found_with_correct_label": round(right / max(total, 1), 4), "test_teeth": total}
    json.dump(mc, open(f"{EXPORT}/pdcnn_boneloss_metrics.json", "w"), indent=2)
    shutil.copy(f"{run_c}/weights/best.pt", f"{EXPORT}/pdcnn_boneloss.pt")
    print(json.dumps(mc, indent=2))

In [ ]:
# 7. Model D: fine-tune the current tooth detector on Aga Khan; keep it only if the held-out test improves
from ultralytics import YOLO
CUR = f"{ROOT}/current/dental_yolov8n.pt"
if not need_D:
    print("Model D already trained or switched off in RUN - skipping")
elif not os.path.exists(CUR):
    print("SKIPPED: upload backend/weights/dental_yolov8n.pt to MyDrive/PerioVision/current/ to run step D")
else:
    aku_src = os.path.dirname(glob.glob(f"{D}/aku/**/folder 1", recursive=True)[0])
    if not os.path.exists("/content/aku_yolo/aku.yaml"):
        !python /content/convert_aku_labelme.py --src "{aku_src}" --out /content/aku_yolo
    before = YOLO(CUR).val(data="/content/aku_yolo/aku.yaml", split="test", imgsz=IMGSZ_D, project=f"{ROOT}/runs", name="aku_before", exist_ok=True)
    run_d = f"{ROOT}/runs/aku_finetune"
    if not os.path.exists(f"{run_d}/weights/best.pt"):
        YOLO(CUR).train(data="/content/aku_yolo/aku.yaml", imgsz=IMGSZ_D, epochs=EPOCHS_D, patience=20, batch=-1, lr0=0.001,
                        freeze=10, fliplr=0.0, project=f"{ROOT}/runs", name="aku_finetune", exist_ok=True, seed=0)
    after = YOLO(f"{run_d}/weights/best.pt").val(data="/content/aku_yolo/aku.yaml", split="test", imgsz=IMGSZ_D, project=f"{ROOT}/runs", name="aku_after", exist_ok=True)
    md_ = {"test_mAP50_before": round(float(before.box.map50), 4), "test_mAP50_after": round(float(after.box.map50), 4),
           "test_mAP50_95_before": round(float(before.box.map), 4), "test_mAP50_95_after": round(float(after.box.map), 4),
           "note": "fliplr=0 because mirroring would swap left / right FDI numbers"}
    md_["adopted"] = md_["test_mAP50_95_after"] > md_["test_mAP50_95_before"]
    json.dump(md_, open(f"{EXPORT}/aku_detector_finetune_metrics.json", "w"), indent=2)
    if md_["adopted"]:
        shutil.copy(f"{run_d}/weights/best.pt", f"{EXPORT}/dental_yolov8n_aku.pt")
    print(json.dumps(md_, indent=2))

In [ ]:
# 8. Summary of everything in the export folder
for f in sorted(glob.glob(f"{EXPORT}/*")):
    print(f"{os.path.basename(f):45s} {os.path.getsize(f)/1e6:8.1f} MB")
for f in sorted(glob.glob(f"{EXPORT}/*metrics.json")):
    print("\n==", os.path.basename(f)); print(open(f).read())

## Done
Download `MyDrive/PerioVision/export_panoramic/` (right-click → Download), extract it into
`C:\Users\<you>\Downloads\export_panoramic`, and tell Claude. Each model is installed only if its held-out
test metrics pass; otherwise panoramic bone loss stays off.